In [1]:
import warnings
warnings.simplefilter(action='ignore')
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from tqdm import tqdm

In [3]:
def read(file_name):
    try:
        fasta = open(file_name, 'r')
        fasta_pairs = {}
        header = ''
        # check every line for a new header
        # if so, set as key and set non header lines as value
        for line in fasta:
            if line[0] == '>':
                header = line.strip('\n')
                fasta_pairs[header] = ''
            else:
                fasta_pairs[header] += line.strip('\n')
        # return dict of fasta entries
        return fasta_pairs
    except IOError:
        print("File \"{}\" could not be opened.".format(file_name))

# FULL RUN

In [ ]:
#accessions= ['at6923','at6929','at7143','at8285','at9104','at9336','at9503','at9578','at9744','at9762','at9806','at9830','at9847','at9852','at9879','at9883','at9900','col0']
accessions= ['col0']
#runs=["full","iso2","gene2"]
runs=["full"]


for run in tqdm(runs):
    tama_liftoff_sqanti = pd.DataFrame()
    tama_liftoff_trans = pd.DataFrame()
    for acc in tqdm(accessions):
        # -------------0-------------
        # make directories
        os.makedirs("{}/{}/isoform".format(run,acc), exist_ok=True)
        os.makedirs("{}/{}/gene".format(run,acc), exist_ok=True)
        # -------------1-------------
        # prepare sqanti output
        sqanti = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}_liftoff_sqanti/{}-tama-vs-liftoff_classification.txt".format(acc,acc), sep="\t")
        sub = sqanti[["isoform","exons","length","structural_category","associated_gene","associated_transcript","subcategory","ORF_seq"]]
        sub["gene"] = sub["isoform"].str.split(".").str[0]

        # -------------3-------------
        # prepare read support
        reads = pd.read_csv("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}.collapsed.tama.v2_read_support.txt".format(acc), sep="\t")
        r_sub = reads[["merge_trans_id","trans_read_count","gene_read_count"]]
        r_sub.columns = ["isoform", "isoform_reads", "gene_reads"]

        # -------------4-------------
        # add accession ID and merge all information
        merged_all = pd.merge(left=sub, right=r_sub, how='left', left_on='isoform', right_on='isoform')
        merged_all['accession'] = acc

        # set dataset to work with
        if run == "iso2":
            merged_all=merged_all[merged_all["isoform_reads"]>1]
        elif run == "gene2":
            merged_all=merged_all[merged_all["gene_reads"]>1]

        # -------------5-------------
        # Generate list of subdataframes (Sqanti ORF analysis)
        genes = list(sub.groupby("gene"))

        gene2ORF = {}
        for i in genes:
            gene2ORF[i[0]]={}
            orfs = i[1]["ORF_seq"].value_counts().index.tolist()
            for orf in orfs:
                gene2ORF[i[0]][orf] = i[1].loc[i[1]["ORF_seq"] == orf]["isoform"].tolist()
        
        
        ORF2iso = pd.DataFrame()
        
        for gene in gene2ORF.keys():
            # get the isoforms per ORF + calculate the number of isoforms per ORF
            isoforms=[]
            n_isoforms=[]
            for orf in gene2ORF[gene].keys():
                isoforms.append(gene2ORF[gene][orf])
                n_isoforms.append(len(gene2ORF[gene][orf]))
            # get read support and sqanti subcategory per isoform
            read_support=[]
            subcategories=[]
            for l in isoforms:
                read_supp_list=[]
                category_list = []
                for iso in l:
                    # read support
                    rs=merged_all.loc[merged_all["isoform"] == iso]["isoform_reads"]
                    read_supp_list.append(int(rs.to_string().split("    ")[1]))
                    # subcategory
                    subcat = merged_all.loc[merged_all["isoform"] == iso]["subcategory"]
                    category_list.append(subcat.to_string().split("    ")[1])
                read_support.append(read_supp_list)
                subcategories.append(category_list)
            gene={"Gene":gene,
                  "n ORFs":len(gene2ORF[gene]),
                  "n Isoforms": sum(n_isoforms),
                  "Sqanti ORFs":list(gene2ORF[gene].keys()),
                  "Isoforms":isoforms,
                  "n_Isoforms_ORF":[len(x) for x in isoforms],
                  "Subcategories":subcategories,
                  "Read_Support":read_support,
                  "Read_Support_ORF":[sum(x) for x in read_support]}
            ORF2iso=pd.concat([ORF2iso, pd.DataFrame([gene])])
        
        tama_liftoff_sqanti = pd.concat([tama_liftoff_sqanti, ORF2iso])

        
        # -------------5-------------
        # Generate list of subdataframes (Transdecoder ORF analysis)
        fasta=read("/ebio/abt6_projects7/dliso/dlis/leon_pipeline/output/v2/collapse_tama/{}_transdecoder/{}_transcripts.fasta.transdecoder.genome.sorted.protein.fasta".format(acc,acc))
        
        trans_df=pd.DataFrame()
        for key in fasta.keys():
            gene = key.split(".")[0].replace(">","")
            iso = key.split(" ")[0].replace(">","").rsplit('.', 1)[0]
            p = key.split(" ")[0].replace(">","").rsplit('.', 1)[1]    
            row={"Gene":gene,
                "isoform":iso,
                "ORF_variant":p,
                "Transdecoder ORF":fasta[key].replace("*","")}
            trans_df=pd.concat([trans_df,pd.DataFrame([row])])
            
        trans_genes = list(trans_df.groupby("Gene"))
        gene2ORF_trans = {}
        for i in trans_genes:
            gene2ORF_trans[i[0]]={}
            orfs = i[1]["Transdecoder ORF"].value_counts().index.tolist()
            for orf in orfs:
                gene2ORF_trans[i[0]][orf] = i[1].loc[i[1]["Transdecoder ORF"] == orf]["isoform"].tolist()
                
        ORF_trans=pd.DataFrame()
        for gene in gene2ORF_trans.keys():
            # get the isoforms per ORF + calculate the number of isoforms per ORF
            isoforms=[]
            n_isoforms=[]
            for orf in gene2ORF_trans[gene].keys():
                isoforms.append(gene2ORF_trans[gene][orf])
                n_isoforms.append(len(gene2ORF_trans[gene][orf]))
            # get read support and sqanti subcategory per isoform
            read_support=[]
            subcategories=[]
            for l in isoforms:
                read_supp_list=[]
                category_list = []
                for iso in l:
                    # read support
                    rs=merged_all.loc[merged_all["isoform"] == iso]["isoform_reads"]
                    read_supp_list.append(int(rs.to_string().split("    ")[1]))
                    # subcategory
                    subcat = merged_all.loc[merged_all["isoform"] == iso]["subcategory"]
                    category_list.append(subcat.to_string().split("    ")[1])
                read_support.append(read_supp_list)
                subcategories.append(category_list)
            gene={"Gene":gene,
                  "n ORFs":len(gene2ORF_trans[gene]),
                  "n Isoforms": sum(n_isoforms),
                  "Sqanti ORFs":list(gene2ORF_trans[gene].keys()),
                  "Isoforms":isoforms,
                  "n_Isoforms_ORF":[len(x) for x in isoforms],
                  "Subcategories":subcategories,
                  "Read_Support":read_support,
                  "Read_Support_ORF":[sum(x) for x in read_support]}
            ORF_trans=pd.concat([ORF_trans, pd.DataFrame([gene])])
        
        tama_liftoff_trans = pd.concat([tama_liftoff_trans, ORF_trans])

            
    ORF2iso.to_csv("Tama_ORF_sqanti_analysis.tsv", sep="\t")
    ORF2iso.to_pickle("Tama_ORF_sqanti_analysis.pkl")
    ORF_trans.to_csv("Tama_ORF_transdecoder_analysis.tsv", sep="\t")
    ORF_trans.to_pickle("Tama_ORF_transdecoder_analysis.pkl")

        
    tama_liftoff_df = pd.concat([tama_liftoff_df, merged_all])
        
    if run == "full":
        tama_liftoff_sqanti.to_pickle("./tama_liftoff_sqanti_orf_{}.pkl".format(run))
        tama_liftoff_sqanti.to_csv("./tama_liftoff_sqanti_orf_{}.tsv".format(run), sep="\t")
        tama_liftoff_trans.to_pickle("./tama_liftoff_transdecoder_orf_{}.tsv".format(run))
        tama_liftoff_trans.to_csv("./tama_liftoff_transdecoder_orf_{}.tsv".format(run), sep="\t")

  0%|                                                                                                                                                                                                                  | 0/1 [00:00<?, ?it/s]